# 01 · Baseline: regresión logística sobre `application_train`

Notebook base del TP1 (Home Credit Default Risk). Objetivo: tener un pipeline de punta a punta funcionando antes de hacer feature engineering.

1. Leer `application_train` y `application_test`
2. Asignar los folds fijos del repo (`data/folds.csv`)
3. Entrenar una logística simple con validación cruzada (métrica: ROC AUC)
4. Generar el archivo de submission para Kaggle

**Requisitos:** los CSV de Kaggle en `data/raw/` y los folds generados (`python -m src.make_folds`).

In [1]:
import sys
from pathlib import Path

# Permite importar el paquete src/ estando parados en notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import ID_COL, TARGET_COL
from src.cv import run_cv
from src.data import attach_folds, load_application
from src.models import build_logistic, get_features
from src.submission import make_submission

## 1. Datos

In [2]:
train = load_application("train")
test = load_application("test")

print(f"train: {train.shape} | test: {test.shape}")
print(f"Tasa de default en train: {train[TARGET_COL].mean():.2%}")
train.head()

train: (307511, 122) | test: (48744, 121)
Tasa de default en train: 8.07%


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Folds

Los folds son fijos y están versionados en git: todos los modelos del grupo se comparan sobre la misma partición.

In [3]:
folds = attach_folds(train)
pd.crosstab(folds, train[TARGET_COL], normalize="index").round(4)

TARGET,0,1
SK_ID_CURR,,
0,0.9193,0.0807
1,0.9193,0.0807
2,0.9193,0.0807
3,0.9193,0.0807
4,0.9193,0.0807


## 3. Modelo baseline

Logística con preprocesamiento mínimo (ver `src/models.py`): mediana + estandarización en numéricas, one-hot en categóricas. Todo va dentro de un `Pipeline`, así que se ajusta solo con el train de cada fold.

In [4]:
X = get_features(train)
y = train[TARGET_COL]
X_test = get_features(test)[X.columns]  # mismo orden de columnas que train

res = run_cv(lambda: build_logistic(C=1.0), X, y, folds, X_test=X_test)

Fold 0: AUC = 0.7399


Fold 1: AUC = 0.7497


Fold 2: AUC = 0.7449


Fold 3: AUC = 0.7497


Fold 4: AUC = 0.7397


AUC OOF total: 0.7448 | media por fold: 0.7448 ± 0.0044


## 4. Submission

In [5]:
path = make_submission(test[ID_COL], res["test_pred"], name="logistica_baseline", cv_auc=res["oof_auc"])
pd.read_csv(path).head()

Submission guardada en /home/claude/tp1-homecredit/submissions/20260921_2225_logistica_baseline_cv0.7448.csv (48744 filas)


,SK_ID_CURR,TARGET
0,100001,0.058312
1,100005,0.217725
2,100013,0.029995
3,100028,0.029055
4,100038,0.119735


## Próximos pasos

* Feature engineering sobre `application` (p. ej. `DAYS_EMPLOYED = 365243` es un valor anómalo, ratios como crédito/ingreso o anualidad/ingreso)
* Agregar información de `bureau`, `previous_application`, `installments_payments`, etc. agregando por `SK_ID_CURR`
* Segundo modelo (árboles / boosting) usando la misma `run_cv`
* Búsqueda de hiperparámetros (p. ej. `C` de la logística) siempre sobre estos folds
* Interpretabilidad: coeficientes de la logística, importancia de variables, SHAP